# Load the libraries

In [1]:
import os
import scanpy as sc
import anndata
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import igraph as ig
import louvain

/Users/apple/miniforge3/envs/scRNA-seq-analysis/lib/python3.9/site-packages/louvain/__init__.py:54: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import get_distribution, DistributionNotFound


## Scanpy configuration

In [2]:
sc.settings.verbosity=3
sc.settings.set_figure_params(dpi=100,facecolor='white',figsize=(8,6))


# Data Manipulation

## Load data in AnnData

In [3]:
adata = sc.read_10x_mtx("data/filtered_feature_bc_matrix/", var_names='gene_symbols', cache=True)
adata.var_names_make_unique()
adata.raw=adata
adata

... reading from cache file cache/data-filtered_feature_bc_matrix-matrix.h5ad


AnnData object with n_obs × n_vars = 11769 × 33538
    var: 'gene_ids', 'feature_types'

## Data preview

In [4]:
print(adata)

print(f"Number of cells: {adata.n_obs}")
print(f"Number of genes: {adata.n_vars}")

print(f"Number of non-zero entries: {adata.X.nnz}")

print(f"First 5 ID cells: {adata.obs_names[:5].tolist()}")
print(f"First 5 gene names: {adata.var_names[:5].tolist()}")


nonzero=(adata.X!=0).sum()
total=adata.shape[0]*adata.shape[1]
print(f"Data sparsity: {(1-nonzero/total):.2%}")



AnnData object with n_obs × n_vars = 11769 × 33538
    var: 'gene_ids', 'feature_types'
Number of cells: 11769
Number of genes: 33538
Number of non-zero entries: 24825783
First 5 ID cells: ['AAACCCAAGCGCCCAT-1', 'AAACCCAAGGTTCCGC-1', 'AAACCCACAGAGTTGG-1', 'AAACCCACAGGTATGG-1', 'AAACCCACATAGTCAC-1']
First 5 gene names: ['MIR1302-2HG', 'FAM138A', 'OR4F5', 'AL627309.1', 'AL627309.3']
Data sparsity: 93.71%


# Quality Control

## Quality Control Metrics and Visualisation

In [5]:
adata.var['mt']=adata.var_names.str.startswith('MT-')
adata.var['ribo']=adata.var_names.str.startswith(('RPS','RPL'))
adata.var['hb']=adata.var_names.str.startswith('HB')



print(f"Mitochondrial genes: {adata.var['mt'].sum()}")
print(f"Ribosomal genes: {adata.var['ribo'].sum()}")
print(f"Hemoglobin genes: {adata.var['hb'].sum()}")



sc.pp.calculate_qc_metrics(adata,qc_vars=['mt','ribo','hb'],percent_top=None,log1p=False,inplace=True)
sc.pl.violin(adata,['n_genes_by_counts','total_counts','pct_counts_mt'],jitter=0.4,multi_panel=True)

fig,ax=plt.subplots(1,2,figsize=(10,4))
sc.pl.scatter(adata,x='total_counts',y='pct_counts_mt',ax=ax[0],show=False)
sc.pl.scatter(adata, x='total_counts', y='n_genes_by_counts', ax=ax[1], show=False)
plt.tight_layout()

if os.path.isdir('figures'):
    plt.savefig("figures/qc_metrics")
else:
    os.makedirs("figures")
    plt.savefig("figures/qc_metrics")
plt.close()



print(f"QC metrics calculated:")
print(f"QC columns in adata.obs: {[col for col in adata.obs.columns if 'total' in col or 'pct' in col or 'n_genes' in col]}")

print(f"\nBasic statistics:")
print(f"Mean genes per cell: {adata.obs['n_genes_by_counts'].mean():.2f}")
print(f"Mean UMIs per cell: {adata.obs['total_counts'].mean():.2f}")
print(f"Mean mitochondrial percentage: {adata.obs['pct_counts_mt'].mean():.2f}%")

if 'pct_counts_ribo' in adata.obs.columns:
    print(f"Mean ribosomal percentage: {adata.obs['pct_counts_ribo'].mean():.2f}%")


Mitochondrial genes: 13
Ribosomal genes: 104
Hemoglobin genes: 13


/Users/apple/miniforge3/envs/scRNA-seq-analysis/lib/python3.9/site-packages/scanpy/plotting/_utils.py:363: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


QC metrics calculated:
QC columns in adata.obs: ['n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'total_counts_ribo', 'pct_counts_ribo', 'total_counts_hb', 'pct_counts_hb']

Basic statistics:
Mean genes per cell: 2109.42
Mean UMIs per cell: 7644.85
Mean mitochondrial percentage: 11.99%
Mean ribosomal percentage: 25.85%


## Quality Control and Filtering

In [6]:
print(f"Before filtering")
print(f"Cells: {adata.n_obs}")
print(f"Genes: {adata.n_vars}")


sc.pp.filter_genes(adata,min_cells=3)
adata=adata[
    (adata.obs['n_genes_by_counts']>500) &
    (adata.obs['n_genes_by_counts']<5000) &
    (adata.obs['pct_counts_mt'] <15) &
    (adata.obs['total_counts']>1000) &
    (adata.obs['total_counts']<30000)
]

adata.obs['qc_pass']=(
    (adata.obs['n_genes_by_counts']>=500) &
    (adata.obs['n_genes_by_counts']<=5000) &
    (adata.obs['pct_counts_mt']<15)
)
failed_qc=adata[~adata.obs['qc_pass']].copy()

#Final cleanup to perform to have at least the basic trimming of the data!!
sc.pp.filter_genes(adata,min_cells=3)
sc.pp.filter_cells(adata,min_genes=500)


Before filtering
Cells: 11769
Genes: 33538
filtered out 13245 genes that are detected in less than 3 cells


/var/folders/83/kdjt3yqd7wlf67c4dgymr3680000gn/T/ipykernel_9514/3450852592.py:15: ImplicitModificationWarning: Trying to modify attribute `.obs` of view, initializing view as actual.
  adata.obs['qc_pass']=(


filtered out 186 genes that are detected in less than 3 cells


### Calculate gene statistics

In [7]:
adata.var['n_cells']=(adata.X>0).sum(axis=0).A1
adata.var['mean_counts']=adata.X.mean(axis=0).A1
adata.var['pct_dropout']=100*(1-adata.var['n_cells']/adata.n_obs)



print("Gene Statistics")
print(f"Mean cells per gene: {adata.var['n_cells'].mean():.2f}")
print(f"Mean count per gene: {adata.var['mean_counts'].mean():.2f}")
print(f"Mean dropout percentage: {adata.var['pct_dropout'].mean():.2f}%")



print(f"\nFiltered data genes")
print(f"Cells: {adata.n_obs}")
print(f"Genes: {adata.n_vars}")

Gene Statistics
Mean cells per gene: 1169.56
Mean count per gene: 0.40
Mean dropout percentage: 88.87%

Filtered data genes
Cells: 10508
Genes: 20107


## Plots for the gene statistics

### Number of genes per cell

In [8]:
fig,ax=plt.subplots(figsize=(5,5))
ax.violinplot([adata.obs['n_genes_by_counts']],showmeans=True,showmedians=True)
ax.set_ylabel('Number of genes')
ax.set_title('Genes per cell')
ax.set_xlabel('Cells')
plt.tight_layout()

if os.path.isdir('figures'):
    plt.savefig("figures/genes_per_cell")
else:
    os.makedirs("figures")
    plt.savefig("figures/genes_per_cell")
plt.close()


### Total UMI counts per cell

In [9]:
fig,ax=plt.subplots(figsize=(5,5))
ax.violinplot(adata.obs['total_counts'],showmedians=True,showmeans=True)
ax.set_ylabel('Total UMI counts')
ax.set_title('UMI counts per cell')
ax.set_xlabel('Cells')
plt.tight_layout()

if os.path.isdir('figures'):
    plt.savefig('figures/umi_per_cell')
else:
    os.makedirs('figures')
    plt.savefig('figures/umi_per_cell')
plt.close()


### Mitochondrial gene percentage

In [10]:
fig,ax=plt.subplots(figsize=(5,5))
ax.violinplot(adata.obs['pct_counts_mt'],showmeans=True,showmedians=True)
ax.set_ylabel('Mithocondrial gene %')
ax.set_title('Mitochondrial gene %')
ax.set_xlabel('Cells')
plt.tight_layout()

if os.path.isdir('figures'):
    plt.savefig('figures/mito_gene_percentage')
else:
    os.makedirs('figures')
    plt.savefig('figures/mito_gene_percentage')
plt.close()


### Genes versus UMIs

In [11]:
fig,ax=plt.subplots(figsize=(5,5))
ax.scatter(adata.obs['total_counts'],adata.obs['n_genes_by_counts'],alpha=0.6,s=1)
ax.set_ylabel('Total UMI counts')
ax.set_xlabel('Number of genes')
ax.set_title('Genes vs UMIs')
plt.tight_layout()

if os.path.isdir('figures'):
    plt.savefig('figures/gene_vs_umi')
else:
    os.makedirs('figures')
    plt.savefig('figures/gene_vs_umi')
plt.close()

### UMIs versus mitochondrial percentage

In [12]:
fig,ax=plt.subplots(figsize=(5,5))
ax.scatter(adata.obs['total_counts'],adata.obs['pct_counts_mt'],alpha=0.6,s=1)
ax.set_xlabel('Total UMI counts')
ax.set_ylabel('Mitochondrial percentage')
ax.set_title('UMI counts vs mitochondrial percentage')
plt.tight_layout()

if os.path.isdir('figures'):
    plt.savefig('figures/umi_vs_mito')
else:
    os.makedirs('figures')
    plt.savefig('figures/umi_vs_mito')
plt.close()

### Ribosomal gene percentage

In [13]:
fig,ax=plt.subplots(figsize=(5,5))
ax.violinplot(adata.obs['pct_counts_ribo'],showmeans=True,showmedians=True)
ax.set_xlabel('Cells')
ax.set_ylabel('Ribosomal gene %')
ax.set_title('Ribosomal gene percentage')
plt.tight_layout()

if os.path.isdir('figures'):
    plt.savefig('figures/ribo_gene_percentage')
else:
    os.makedirs('figures')
    plt.savefig('figures/ribo_gene_percentage')
plt.close()

### Summary statistics

In [14]:
print("\nDetailed statistics")
for metric in ['n_genes_by_counts','total_counts','pct_counts_mt','pct_counts_ribo']:
    if metric in adata.obs.columns:
        data=adata.obs[metric]
        print(f"{metric}")
        print(f"Mean: {data.mean():.2f}")
        print(f"Median: {data.median():.2f}")
        print(f"Std: {data.std():.2f}")
        print(f"Min: {data.min():.2f}")
        print(f"Max: {data.max():.2f}")
        print()


Detailed statistics
n_genes_by_counts
Mean: 2238.32
Median: 1969.00
Std: 818.91
Min: 501.00
Max: 4986.00

total_counts
Mean: 8088.38
Median: 6806.00
Std: 4109.33
Min: 1001.00
Max: 29732.00

pct_counts_mt
Mean: 9.06
Median: 8.88
Std: 2.14
Min: 0.39
Max: 15.00

pct_counts_ribo
Mean: 27.66
Median: 28.32
Std: 10.81
Min: 0.72
Max: 53.15



# Normalisation, scaling and HVG

In [15]:
sc.pp.normalize_total(adata,target_sum=1e4)
sc.pp.log1p(adata)

sc.pp.highly_variable_genes(adata,min_mean=0.0125,max_mean=3,min_disp=0.5)

plt.figure(figsize=(8,4))
sc.pl.highly_variable_genes(adata,show=False)
plt.title("Highly Variiable Genes")

if os.path.isdir('figures'):
    plt.savefig('figures/hvg')
else:
    os.makedirs('figures')
    plt.savefig('figures/hvg')
plt.close()

adata=adata[:,adata.var.highly_variable]
sc.pp.regress_out(adata,['total_counts','pct_counts_mt'])
sc.pp.scale(adata,max_value=10)

normalizing counts per cell
    finished (0:00:00)
extracting highly variable genes
    finished (0:00:01)
--> added
    'highly_variable', boolean vector (adata.var)
    'means', float vector (adata.var)
    'dispersions', float vector (adata.var)
    'dispersions_norm', float vector (adata.var)


/Users/apple/miniforge3/envs/scRNA-seq-analysis/lib/python3.9/site-packages/pandas/core/util/hashing.py:330: RuntimeWarning: invalid value encountered in cast
  vals.astype(str).astype(object), hash_key, encoding


regressing out ['total_counts', 'pct_counts_mt']


/Users/apple/miniforge3/envs/scRNA-seq-analysis/lib/python3.9/site-packages/scanpy/preprocessing/_simple.py:668: UserWarning: Received a view of an AnnData. Making a copy.
  view_to_actual(adata)


    sparse input is densified and may lead to high memory use
    finished (0:00:17)


## Dimensionality reduction

### PCA variance ration

In [16]:
sc.tl.pca(adata, svd_solver='arpack', n_comps=50)

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

variance_ratio = adata.uns['pca']['variance_ratio']
axes[0,0].plot(range(1, len(variance_ratio)+1), variance_ratio, 'bo-', markersize=3)
axes[0,0].set_xlabel('Principal component')
axes[0,0].set_ylabel('Variance ratio')
axes[0,0].set_title('PCA variance ratio')
axes[0,0].set_yscale('log')


pca_coords = adata.obsm['X_pca']
scatter1 = axes[0,1].scatter(pca_coords[:, 0], pca_coords[:, 1], 
                            c=adata.obs['total_counts'], s=1, alpha=0.7, cmap='viridis')
axes[0,1].set_xlabel('PC1')
axes[0,1].set_ylabel('PC2')
axes[0,1].set_title('PCA - Total counts')
plt.colorbar(scatter1, ax=axes[0,1])


scatter2 = axes[1,0].scatter(pca_coords[:, 0], pca_coords[:, 1], 
                            c=adata.obs['n_genes_by_counts'], s=1, alpha=0.7, cmap='viridis')
axes[1,0].set_xlabel('PC1')
axes[1,0].set_ylabel('PC2')
axes[1,0].set_title('PCA - Number of genes')
plt.colorbar(scatter2, ax=axes[1,0])



scatter3 = axes[1,1].scatter(pca_coords[:, 0], pca_coords[:, 1], 
                            c=adata.obs['pct_counts_mt'], s=1, alpha=0.7, cmap='viridis')
axes[1,1].set_xlabel('PC1')
axes[1,1].set_ylabel('PC2')
axes[1,1].set_title('PCA - Mitochondrial %')
plt.colorbar(scatter3, ax=axes[1,1])

plt.tight_layout()

if os.path.isdir('figures'):
    plt.savefig('figures/pca')
else:
    os.makedirs('figures')
    plt.savefig('figures/pca')


print(f"PCA completed:")
print(f"  Number of principal components: {adata.obsm['X_pca'].shape[1]}")
print(f"  Variance explained by first PC: {adata.uns['pca']['variance_ratio'][0]:.3f}")
print(f"  Variance explained by first 10 PCs: {adata.uns['pca']['variance_ratio'][:10].sum():.3f}")
print(f"  Variance explained by first 50 PCs: {adata.uns['pca']['variance_ratio'][:50].sum():.3f}")

n_top_genes = 5
for pc in range(3):  # First 3 PCs
    pc_loadings = adata.varm['PCs'][:, pc]
    top_genes_idx = np.argsort(np.abs(pc_loadings))[-n_top_genes:]
    top_genes = adata.var_names[top_genes_idx]
    loadings_values = pc_loadings[top_genes_idx]
    print(f"\nTop {n_top_genes} genes for PC{pc+1}:")
    for gene, loading in zip(top_genes, loadings_values):
        print(f"  {gene}: {loading:.3f}")

computing PCA
    with n_comps=50
    finished (0:00:09)
PCA completed:
  Number of principal components: 50
  Variance explained by first PC: 0.067
  Variance explained by first 10 PCs: 0.191
  Variance explained by first 50 PCs: 0.244

Top 5 genes for PC1:
  FGL2: 0.082
  CTSS: 0.083
  CST3: 0.085
  MNDA: 0.085
  FCN1: 0.086

Top 5 genes for PC2:
  IGHD: 0.100
  BANK1: 0.102
  IGHM: 0.106
  MS4A1: 0.107
  CD79A: 0.111

Top 5 genes for PC3:
  PPBP: 0.112
  GNG11: 0.113
  PF4: 0.114
  CAVIN2: 0.114
  TUBB1: 0.115


### Clustering

In [17]:
sc.pp.neighbors(adata, n_neighbors=10, n_pcs=40)
sc.tl.umap(adata, random_state=42)

sc.tl.leiden(
    adata,
    resolution=0.15,
    random_state=42,
    flavor="igraph",
    directed=False,
    n_iterations=2,
    key_added='leiden_0.15'
)

sc.tl.leiden(adata, resolution=0.5, key_added='leiden_0.5')
sc.tl.leiden(adata, resolution=0.8, key_added='leiden_0.8')
sc.tl.leiden(adata, resolution=1.0, key_added='leiden_1.0')

sc.tl.louvain(
    adata,
    flavor="igraph",
    directed=False,
    random_state=42
)

for resolution in ['leiden_0.15', 'leiden_0.5', 'leiden_0.8', 'leiden_1.0']:
    n_clusters = len(adata.obs[resolution].unique())
    print(f"  {resolution}: {n_clusters} clusters")

adata.obs['clusters'] = adata.obs['leiden_0.15']

print(adata.obs['leiden_0.15'].value_counts().sort_index())
print(adata.obs['louvain'].value_counts().sort_index())


computing neighbors
    using 'X_pca' with n_pcs = 40


/Users/apple/miniforge3/envs/scRNA-seq-analysis/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


    finished: added to `.uns['neighbors']`
    `.obsp['distances']`, distances for each pair of neighbors
    `.obsp['connectivities']`, weighted adjacency matrix (0:00:36)
computing UMAP
    finished: added
    'X_umap', UMAP coordinates (adata.obsm)
    'umap', UMAP parameters (adata.uns) (0:00:06)
running Leiden clustering
    finished: found 11 clusters and added
    'leiden_0.15', the cluster labels (adata.obs, categorical) (0:00:00)
running Leiden clustering


/var/folders/83/kdjt3yqd7wlf67c4dgymr3680000gn/T/ipykernel_9514/460595946.py:14: FutureWarning: In the future, the default backend for leiden will be igraph instead of leidenalg.

 To achieve the future defaults please pass: flavor="igraph" and n_iterations=2.  directed must also be False to work with igraph's implementation.
  sc.tl.leiden(adata, resolution=0.5, key_added='leiden_0.5')


    finished: found 19 clusters and added
    'leiden_0.5', the cluster labels (adata.obs, categorical) (0:00:01)
running Leiden clustering
    finished: found 23 clusters and added
    'leiden_0.8', the cluster labels (adata.obs, categorical) (0:00:00)
running Leiden clustering
    finished: found 25 clusters and added
    'leiden_1.0', the cluster labels (adata.obs, categorical) (0:00:00)
running Louvain clustering
    finished: found 21 clusters and added
    'louvain', the cluster labels (adata.obs, categorical) (0:00:00)
  leiden_0.15: 11 clusters
  leiden_0.5: 19 clusters
  leiden_0.8: 23 clusters
  leiden_1.0: 25 clusters
leiden_0.15
0     3121
1      956
2      312
3     3055
4      610
5     1392
6      340
7       82
8      346
9       95
10     199
Name: count, dtype: int64
louvain
0     1663
1      181
2     1722
3      560
4      950
5      822
6     1031
7      338
8       83
9      150
10     688
11     349
12     108
13     146
14     435
15     440
16     138
17     33

### UMAP

In [18]:
fig, axes = plt.subplots(2, 3, figsize=(18, 12))
fig.suptitle('UMAP visualization', fontsize=16)


umap_coords = adata.obsm['X_umap']
clusters = adata.obs['clusters'].astype(str)
unique_clusters = sorted(clusters.unique(), key=lambda x: int(x))


colors = plt.cm.tab20(np.linspace(0, 1, len(unique_clusters)))
cluster_colors = {cluster: colors[i] for i, cluster in enumerate(unique_clusters)}

for cluster in unique_clusters:
    mask = clusters == cluster
    axes[0,0].scatter(umap_coords[mask, 0], umap_coords[mask, 1], 
                     c=[cluster_colors[cluster]], label=f'Cluster {cluster}', s=1, alpha=0.7)
axes[0,0].set_xlabel('UMAP1')
axes[0,0].set_ylabel('UMAP2')
axes[0,0].set_title('Clusters (Leiden 0.15)')
axes[0,0].legend(bbox_to_anchor=(1.05, 1), loc='upper left', markerscale=5)



scatter1 = axes[0,1].scatter(umap_coords[:, 0], umap_coords[:, 1], 
                            c=adata.obs['total_counts'], s=1, alpha=0.7, cmap='viridis')
axes[0,1].set_xlabel('UMAP1')
axes[0,1].set_ylabel('UMAP2')
axes[0,1].set_title('Total counts')
plt.colorbar(scatter1, ax=axes[0,1])



scatter2 = axes[0,2].scatter(umap_coords[:, 0], umap_coords[:, 1], 
                            c=adata.obs['n_genes_by_counts'], s=1, alpha=0.7, cmap='viridis')
axes[0,2].set_xlabel('UMAP1')
axes[0,2].set_ylabel('UMAP2')
axes[0,2].set_title('Number of genes')
plt.colorbar(scatter2, ax=axes[0,2])



scatter3 = axes[1,0].scatter(umap_coords[:, 0], umap_coords[:, 1], 
                            c=adata.obs['pct_counts_mt'], s=1, alpha=0.7, cmap='viridis')
axes[1,0].set_xlabel('UMAP1')
axes[1,0].set_ylabel('UMAP2')
axes[1,0].set_title('Mitochondrial %')
plt.colorbar(scatter3, ax=axes[1,0])



scatter4 = axes[1,1].scatter(umap_coords[:, 0], umap_coords[:, 1], 
                            c=adata.obs['pct_counts_ribo'], s=1, alpha=0.7, cmap='viridis')
axes[1,1].set_xlabel('UMAP1')
axes[1,1].set_ylabel('UMAP2')
axes[1,1].set_title('Ribosomal %')
plt.colorbar(scatter4, ax=axes[1,1])



clusters_05 = adata.obs['leiden_0.5'].astype(str)
unique_clusters_05 = sorted(clusters_05.unique(), key=lambda x: int(x))
colors_05 = plt.cm.tab20(np.linspace(0, 1, len(unique_clusters_05)))
cluster_colors_05 = {cluster: colors_05[i] for i, cluster in enumerate(unique_clusters_05)}

for cluster in unique_clusters_05:
    mask = clusters_05 == cluster
    axes[1,2].scatter(umap_coords[mask, 0], umap_coords[mask, 1], 
                     c=[cluster_colors_05[cluster]], s=1, alpha=0.7)
axes[1,2].set_xlabel('UMAP1')
axes[1,2].set_ylabel('UMAP2')
axes[1,2].set_title('Clusters (Leiden 0.5)')

plt.tight_layout()
if os.path.isdir('figures'):
    plt.savefig('figures/umap')
else:
    os.makedirs('figures')
    plt.savefig('figures/umap')
print(f"UMAP coordinates shape: {adata.obsm['X_umap'].shape}")

UMAP coordinates shape: (10508, 2)


## Differential Expression

### Identify Marker Genes

In [19]:
sc.tl.rank_genes_groups(
    adata, 
    groupby='leiden_0.15',
    method='wilcoxon',
    key_added='rank_genes_groups',
    pts=True
)

sc.pl.rank_genes_groups(adata, n_genes=5,sharey=False,show=False,save="_rank_genes.png")



marker_df = sc.get.rank_genes_groups_df(adata, group='0')  # change group as needed


ranking genes
    finished: added to `.uns['rank_genes_groups']`
    'names', sorted np.recarray to be indexed by group ids
    'scores', sorted np.recarray to be indexed by group ids
    'logfoldchanges', sorted np.recarray to be indexed by group ids
    'pvals', sorted np.recarray to be indexed by group ids
    'pvals_adj', sorted np.recarray to be indexed by group ids (0:00:51)


### Visualize and export marker genes

In [20]:
# Access result from AnnData object
result = adata.uns['rank_genes_groups']
groups = result['names'].dtype.names

# Build dataframe
marker_df = pd.DataFrame({
    f"{group}_{key}": result[key][group]
    for group in groups for key in ['names', 'pvals_adj', 'logfoldchanges']
})

# Show top genes
print(marker_df.head(10))

# Save to CSV
marker_df.to_csv("marker_genes_per_cluster.csv")

# Top 5 marker genes per cluster
sc.pl.rank_genes_groups_stacked_violin(
    adata,
    n_genes=5,
    groupby='leiden_0.15',
    use_raw=True,
    show=True,
    save="marker_genes.png"
)

  0_names  0_pvals_adj  0_logfoldchanges 1_names    1_pvals_adj  \
0    TRAC          0.0          6.656347    NKG7   0.000000e+00   
1    IL7R          0.0          8.692817    CCL5   0.000000e+00   
2    CD3D          0.0          4.939792    GZMA   0.000000e+00   
3    LDHB          0.0          6.247232   KLRG1   0.000000e+00   
4    CD3G          0.0          4.180652    GZMK   0.000000e+00   
5    CD3E          0.0          4.279090    CST7   0.000000e+00   
6    TCF7          0.0          3.993860    CTSW   0.000000e+00   
7   RPS29          0.0         28.886854    IL32   0.000000e+00   
8   RPS27          0.0         53.067001   TRGC2  1.683513e-271   
9     LTB          0.0          8.891593   KLRB1  1.821879e-255   

   1_logfoldchanges 2_names    2_pvals_adj  2_logfoldchanges     3_names  ...  \
0         11.805905   RPS24  1.002961e-172         93.666603      S100A8  ...   
1         12.122824  MT-CO3  1.943277e-167        244.052811      S100A9  ...   
2          7.785745

/Users/apple/miniforge3/envs/scRNA-seq-analysis/lib/python3.9/site-packages/scanpy/plotting/_utils.py:363: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Cell type annotation

### Define marker gene signatures and visualize markers

In [21]:
pbmc_markers = {
    'CD4+ T cells': ['CD4', 'IL7R', 'CCR7'],
    'CD8+ T cells': ['CD8A', 'CD8B', 'GZMK'],
    'Cytotoxic T/NK Cells': ['GZMA', 'GZMB', 'NKG7', 'KLRB1'],
    'NK cells': ['GNLY', 'NKG7', 'FGFBP2'],
    'B cells': ['MS4A1', 'CD79A', 'CD79B'],
    'CD14+ Monocytes': ['CD14', 'LYZ', 'CD68'],
    'FCGR3A+ Monocytes': ['FCGR3A', 'MS4A7'],
    'Dendritic Cells': ['FCER1A', 'CST3'],
    'Megakaryocytes': ['PPBP', 'PF4'],
    'Progenitor-like': ['SOX4', 'HMGB1', 'RPL15']  # These are approximate, often dataset-specific
}


sc.pl.dotplot(adata, var_names=pbmc_markers, groupby='leiden_0.15', use_raw=True,save="marker_gene_sign.png")
sc.pl.matrixplot(adata,
    var_names=pbmc_markers,
    groupby='leiden_0.15',
    use_raw=True,
    standard_scale='var',
    cmap='Reds',
    save="marker_gene_sign.png"
)

/Users/apple/miniforge3/envs/scRNA-seq-analysis/lib/python3.9/site-packages/scanpy/plotting/_utils.py:363: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
/Users/apple/miniforge3/envs/scRNA-seq-analysis/lib/python3.9/site-packages/scanpy/plotting/_utils.py:363: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### Assign cell types to clusters

In [27]:
cluster_annotation = {
    '0': 'CD4+ T cells',
    '1': 'Cytotoxic T/NK Cells', 
    '2': 'Dendritic Cells',
    '3': 'Monocytes',
    '4': 'FCGR3A+ Monocytes',
    '5': 'NK Cells',
    '6': 'B Cells',
    '7': 'CD8+ T cells',
    '8': 'Megakaryocytes',
    '9': 'Progenitor-like',
    '10':'Other'
}

adata.obs['cell_type'] = adata.obs['leiden_0.15'].map(cluster_annotation)

sc.pl.umap(adata, color='cell_type', legend_loc='on data', 
           legend_fontsize=10, legend_fontoutline=2, frameon=False,save='_clusters_assigned.png')

/Users/apple/miniforge3/envs/scRNA-seq-analysis/lib/python3.9/site-packages/scanpy/plotting/_utils.py:363: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### Validate Annotation

In [28]:
sc.pl.umap(
    adata,
    color=['LYZ', 'RPL15', 'PPBP'],
    use_raw=True,
    vmin=0,
    vmax=100,
    cmap='Reds',
    size=20,
    save='_val_ann.png'
)


/Users/apple/miniforge3/envs/scRNA-seq-analysis/lib/python3.9/site-packages/scanpy/plotting/_utils.py:363: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### Trajectory Analysis

In [31]:
sc.tl.paga(adata, groups='leiden_0.15')
sc.pl.paga(adata,show=False,save=".png")
sc.tl.umap(adata, init_pos='paga', random_state=42)



adata.obs['cell_type']=adata.obs['leiden_0.15'].astype(str).map(cluster_annotation)
sc.tl.paga(adata,groups='cell_type')
sc.pl.paga(adata,show=False,save='_cell_type.png')

running PAGA
    finished: added
    'paga/connectivities', connectivities adjacency (adata.uns)
    'paga/connectivities_tree', connectivities subtree (adata.uns) (0:00:00)
--> added 'pos', the PAGA positions (adata.uns['paga'])
computing UMAP
    finished: added
    'X_umap', UMAP coordinates (adata.obsm)
    'umap', UMAP parameters (adata.uns) (0:00:06)
running PAGA
    finished: added
    'paga/connectivities', connectivities adjacency (adata.uns)
    'paga/connectivities_tree', connectivities subtree (adata.uns) (0:00:00)
--> added 'pos', the PAGA positions (adata.uns['paga'])


<Axes: >